# Router experiments — train, validate, use

Minimal harness for testing configs. Edit the **config** cell, then Run All.
Every knob is an existing `RouterExperiment` / `StrategyRouter` flag.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from hybrid_search_rrf_dataset.router import (
    QueryEncoder,
    Representation,
    RouterExperiment,
    StrategyRouter,
)

# ---- config: edit and Run All ----
REPRESENTATION = Representation.ENGINEERED  # ENGINEERED | EMBEDDING | BOTH
ENCODER = None           # QueryEncoder() for EMBEDDING/BOTH (downloads e5 once)
PROTOCOL = 'random_within_lane'  # or 'holdout_lane' (train/use cells only)
ALL_ROWS = 'decisive'     # 'decisive' ~4K clear wins | 'recommended' ~12K all real winners | 'all' ~37K, tied/zero rows as negatives
MAX_CLASS_SHARE = None    # e.g. 0.5: downsample so no winner class exceeds that share

In [4]:
# 1 — train
from hybrid_search_rrf_dataset.router import _derive_engineered, _margin, _routes_differ

exp = RouterExperiment(encoder=ENCODER)
train, test = exp.split(PROTOCOL)

router = StrategyRouter(REPRESENTATION, encoder=ENCODER).fit(
    train, all_rows=ALL_ROWS, max_class_share=MAX_CLASS_SHARE
)
router.tune_thresholds(train)
print('thresholds (dense, sparse):', tuple(round(t, 3) for t in router.thresholds))

# coefficients + support: `fires` = substrate rows where the feature is nonzero.
# A big weight with tiny support is a scaling artifact, not evidence.
mode = {False: 'decisive', True: 'recommended'}.get(ALL_ROWS, ALL_ROWS)
fit_rows = (
    train if mode == 'all'
    else train[_routes_differ(train)] if mode == 'recommended'
    else train[_margin(train) >= router.decisive_margin]
)
eng = _derive_engineered(fit_rows)
coef = router.coefficients().set_index('feature')
coef['fires'] = [(eng[f] > 0).sum() if f in eng.columns else None for f in coef.index]
coef.round(3).nlargest(8, 'sparse_weight')

thresholds (dense, sparse): (0.2, 0.75)


,dense_weight,sparse_weight,fires
feature,,,
length.length_chars,-1.051,1.037,4079
structured_identifiers.bic,-0.662,0.498,425
structured_identifiers.postal_code,-0.308,0.327,22
structured_identifiers.email,-0.233,0.278,4
syntactic_depth.nesting_depth,-0.308,0.265,4021
structured_identifiers.betting_odds,-0.201,0.235,19
morphology.word_variation_share,-0.175,0.208,3440
sentence_markers.acronym,-0.190,0.185,615


In [51]:
# 2 — validate: six-column table, both protocols, this config
cols = [
    'protocol', 'representation', 'all_rows', 'max_class_share', 'n_test_decisive',
    'const_dense_only', 'const_pure_rrf', 'const_sparse_only',
    'oracle', 'router', 'headroom_captured', 't_dense', 't_sparse',
]
result = exp.run(
    representations=[REPRESENTATION],
    all_rows=ALL_ROWS,
    max_class_share=MAX_CLASS_SHARE,
)
result[cols].round(3)

holdout_lane·engineered: 100%|██████████| 2/2 [00:00<00:00, 20.53it/s, headroom=0.031, n=206]


,protocol,representation,all_rows,max_class_share,n_test_decisive,const_dense_only,const_pure_rrf,const_sparse_only,oracle,router,headroom_captured,t_dense,t_sparse
0,random_within_lane,engineered,decisive,None,1021,0.631,0.208,0.367,0.975,0.709,0.226,0.20,0.70
1,holdout_lane,engineered,decisive,None,206,0.607,0.185,0.430,1.000,0.619,0.031,0.25,0.75


In [6]:
# 3 — use: route your own queries with the §1 router (needs en_core_web_sm)
my_queries = [
    'Who likes Curling?',
    'what are the side effects of DHA',
    'CVE-2021-44228 log4j remote code execution',
    'http://localhost.com',
]
for q in my_queries:
    e = router.explain(q)
    print(f"{e['route']!s:12s} p_dense={e['p_dense']:.2f} p_sparse={e['p_sparse']:.2f}  {q}")

dense_only   p_dense=0.60 p_sparse=0.40  Who likes Curling?
dense_only   p_dense=0.67 p_sparse=0.33  what are the side effects of DHA
dense_only   p_dense=0.48 p_sparse=0.51  CVE-2021-44228 log4j remote code execution
dense_only   p_dense=0.31 p_sparse=0.67  http://localhost.com


In [7]:
# 4 — serve: refit on ALL labelled data with this config (no held-out split).
# This is the model you'd ship; its numbers are NOT comparable to §2's,
# which must hold data out to stay an honest measurement.
data = exp.load()
served = StrategyRouter(REPRESENTATION, encoder=ENCODER, delta=0.14).fit(
    data, all_rows='decisive', max_class_share=MAX_CLASS_SHARE
)
# served.tune_thresholds(data)
print('serving thresholds (dense, sparse):', tuple(round(t, 3) for t in served.thresholds))
for q in my_queries:
    e = served.explain(q)
    print(f"{e['route']!s:12s} p_dense={e['p_dense']:.2f} p_sparse={e['p_sparse']:.2f}  {q}")

serving thresholds (dense, sparse): (0.5, 0.5)
dense_only   p_dense=0.59 p_sparse=0.41  Who likes Curling?
dense_only   p_dense=0.64 p_sparse=0.34  what are the side effects of DHA
pure_rrf     p_dense=0.48 p_sparse=0.51  CVE-2021-44228 log4j remote code execution
sparse_only  p_dense=0.31 p_sparse=0.65  http://localhost.com


In [ ]:
# probe the served model — edit the list and rerun
probes = [
    'qdrant_client.http.models.FormulaQuery',
    'qdrant HNSW ef_construct default value',
    'error code 429 rate limit exceeded API',
    'docker-compose.yml volume mount permissions',
    'RTX 4090 vs A100 fp16 throughput benchmark',
    'python list comprehension syntax',
    'How can I make repeated code easier to maintain in Python?',
    'Why would a website tell me a request is malformed instead of unauthorized?',
    'dylans article about qdrant search',
    'qdrant search',
    '/dylan/neil/jenny',
    'ThinkPad X1 broken screen replacement',
    'ThindkPadX1 is genuinely great laptip',
]
for q in probes:
    e = served.explain(q)
    print(f"{e['route']!s:12s} p_dense={e['p_dense']:.2f} "
          f"p_sparse={e['p_sparse']:.2f}  {q}")


In [110]:
golden_set_auto_fusion = [
    {"query": "who founded apple?", "expected_hi": 2, "expected_lo": 0},
    {"query": "how does photosynthesis work in plants", "expected_hi": 2, "expected_lo": 0},
    {"query": "explain quicksort", "expected_hi": 2, "expected_lo": 0},
    {"query": "best laptop for college students 2024", "expected_hi": 3, "expected_lo": 0},
    {"query": "comment volent les oiseaux", "expected_hi": 2, "expected_lo": 0},
    {"query": "como aprender a programar en rust", "expected_hi": 2, "expected_lo": 0},
    {"query": "tell me about dogs", "expected_hi": 2, "expected_lo": 0},
    {"query": "hey can you help me out", "expected_hi": 2, "expected_lo": 0},
    {"query": "550e8400-e29b-41d4-a716-446655440000", "expected_hi": 9, "expected_lo": 8},
    {"query": "00000000-0000-0000-0000-000000000000", "expected_hi": 9, "expected_lo": 8},
    {"query": "ERR_CONNECTION_RESET", "expected_hi": 9, "expected_lo": 8},
    {"query": "ENOENT", "expected_hi": 9, "expected_lo": 7},
    {"query": "HTTP 502", "expected_hi": 9, "expected_lo": 6},
    {"query": "v1.2.3 changelog", "expected_hi": 9, "expected_lo": 6},
    {"query": "Python 3.11.4 release notes", "expected_hi": 8, "expected_lo": 5},
    {"query": "a3f5d8b9e12c4d56789abcdef0123456", "expected_hi": 9, "expected_lo": 8},
    {"query": "/etc/nginx/nginx.conf", "expected_hi": 9, "expected_lo": 7},
    {"query": "B07XJ8C8F5", "expected_hi": 9, "expected_lo": 7},
    {"query": "GPT-3", "expected_hi": 8, "expected_lo": 5},
    {"query": "BERT model paper", "expected_hi": 7, "expected_lo": 4},
    {"query": "ThinkPad X1 broken screen replacement", "expected_hi": 6, "expected_lo": 3},
    {"query": "iPhone 15 Pro Max battery life", "expected_hi": 6, "expected_lo": 3},
    {"query": "kubernetes pod CrashLoopBackOff", "expected_hi": 8, "expected_lo": 5},
    {"query": "why does my Java program throw NullPointerException at line 42", "expected_hi": 6, "expected_lo": 3},
    {"query": "C++ undefined reference to vtable", "expected_hi": 8, "expected_lo": 5},
    {"query": "how to fix broken screen on my Lenovo ThinkPad X1", "expected_hi": 5, "expected_lo": 2},
    {"query": "linux", "expected_hi": 3, "expected_lo": 1},
    {"query": "covid", "expected_hi": 3, "expected_lo": 1},
    {"query": "the", "expected_hi": 2, "expected_lo": 0},
    {"query": "hello", "expected_hi": 2, "expected_lo": 0},
]

In [ ]:
# router vs the production auto-fusion bands (0-2 dense, 3-6 rrf, 7-9 sparse):
# agree = the router's route falls inside the band the classifier would allow
import pandas as pd

from hybrid_search_rrf_dataset.router import _production_route

rows = []
for case in golden_set_auto_fusion:
    band = {_production_route(s)
            for s in range(case["expected_lo"], case["expected_hi"] + 1)}
    route = served.explain(case["query"])["route"]
    rows.append({
        "query": case["query"],
        "router": route.value,
        "autofusion band": "..".join(sorted(r.value for r in band)),
        "agree": route in band,
    })
frame = pd.DataFrame(rows)
print(f"router inside the auto-fusion band on {frame['agree'].sum()}/{len(frame)}")
frame[~frame["agree"]]


In [ ]:
# 5 — margin hedge: rrf when |p_dense − p_sparse| < delta.
# Delta is tuned on the train routes_differ frame (never on test),
# then judged on held-out decisive rows against both baselines.
# If the best delta is 0.0, the data says the hedge doesn't pay.
import numpy as np

from hybrid_search_rrf_dataset.router import _mean_objective, _route_from_probs

tune_frame = train[_routes_differ(train)]
p_d, p_s = router._probabilities(tune_frame)
deltas = np.round(np.arange(0.0, 0.32, 0.002), 2)[::-1]
tune_scores = [
    _mean_objective(tune_frame, _route_from_probs(p_d, p_s, 0.5, 0.5, delta=d))
    for d in deltas
]
best_delta = float(deltas[int(np.argmax(tune_scores))])
print(f'best delta on train: {best_delta:.2f} '
      f'(objective {max(tune_scores):.4f} vs {tune_scores[-1]:.4f} at delta=0)')

decisive_test = test[_margin(test) >= router.decisive_margin]
pt_d, pt_s = router._probabilities(decisive_test)
configs = {
    'tuned thresholds, delta=0': (*router.thresholds, 0.0),
    'symmetric (0.5, 0.5), delta=0': (0.5, 0.5, 0.0),
    f'symmetric + delta={best_delta:.2f}': (0.5, 0.5, best_delta),
}
for label, (td, ts, d) in configs.items():
    routes = _route_from_probs(pt_d, pt_s, td, ts, delta=d)
    score = _mean_objective(decisive_test, routes)
    n_rrf = sum(r.value == 'pure_rrf' for r in routes)
    print(f'{label:32s} held-out: {score:.3f}  (rrf fired on {n_rrf}/{len(routes)})')


# 6 — Acceptability heads vs argmax router (SPEC d60)

The d60 form: three binary heads trained on `ok_* = score >= oracle − 0.3`
(hit parity), serving the cheapest route whose P(ok) clears the threshold.
Trains on **answerable** rows only — all_zero rows carry null labels — so
the 15K tied rows finally contribute (as sparse/dense/rrf positives) instead
of being filtered out as winnerless.

Read the table against the `serve oracle (view)` row — the cost-aware
ceiling. Two things to watch:

- **objective columns**: the heads router must hold the argmax router's
  quality (the objective ignores cost, so ties score identically whichever
  route serves them).
- **mean cost + route mix**: this is where the two forms should actually
  differ — the heads router is *trained* to prefer cheap-when-tied, the
  argmax router only ever saw winners.

In [8]:
import pandas as pd

from hybrid_search_rrf_dataset.labels import AcceptabilityLabels
from hybrid_search_rrf_dataset.router import AcceptabilityRouter

# same split, same representation as §1 — the label form is the only variable
acc = AcceptabilityRouter(REPRESENTATION, encoder=ENCODER).fit(train)
print(f"tolerance {acc.tolerance} (hit parity) | threshold {acc.threshold}")

view = AcceptabilityLabels(test).frame()
answerable = view[view["serve"].notna()].reset_index(drop=True)
print(f"test: {len(answerable):,} answerable of {len(view):,} "
      f"({(view['serve'].isna()).sum():,} all_zero excluded)")

acc_routes = acc.predict_routes(answerable)
argmax_routes = router.predict_routes(answerable)   # the §1 router

tolerance 0.3 (hit parity) | threshold 0.5
test: 7,618 answerable of 9,225 (1,607 all_zero excluded)


In [9]:
import numpy as np

from hybrid_search_rrf_dataset.fusion import SERVING_COST, StrategyName
from hybrid_search_rrf_dataset.router import _margin, _mean_objective

decisive_mask = (_margin(answerable) >= router.decisive_margin).to_numpy()
serve_oracle = [StrategyName(s) for s in answerable["serve"]]


def readout(policy: str, routes) -> dict:
    routes = list(routes)
    on_decisive = [r for r, m in zip(routes, decisive_mask) if m]
    mix = pd.Series([r.value for r in routes]).value_counts(normalize=True)
    return {
        "policy": policy,
        "objective (answerable)": _mean_objective(answerable, routes),
        "objective (decisive)": _mean_objective(
            answerable[decisive_mask], on_decisive
        ),
        "serve agreement": float(np.mean(
            [r == s for r, s in zip(routes, serve_oracle)]
        )),
        "mean cost": float(np.mean([SERVING_COST[r] for r in routes])),
        **{f"% {s.value}": float(mix.get(s.value, 0.0)) for s in StrategyName},
    }


table = pd.DataFrame([
    readout("serve oracle (view)", serve_oracle),
    readout("acceptability heads", acc_routes),
    readout("argmax router (§1)", argmax_routes),
    *(readout(f"const {s.value}", [s] * len(answerable)) for s in StrategyName),
])
table.round(3)

,policy,objective (answerable),objective (decisive),serve agreement,mean cost,% dense_only,% pure_rrf,% sparse_only
0,serve oracle (view),0.791,0.975,1.000,0.235,0.223,0.006,0.771
1,acceptability heads,0.694,0.639,0.508,0.581,0.536,0.022,0.442
2,argmax router (§1),0.717,0.714,0.295,0.916,0.915,0.001,0.085
3,const dense_only,0.691,0.631,0.223,1.000,1.000,0.000,0.000
4,const pure_rrf,0.694,0.208,0.006,2.000,0.000,1.000,0.000
5,const sparse_only,0.589,0.367,0.771,0.000,0.000,0.000,1.000


In [10]:
# the serving surface, probed: three P(ok) per query, cheapest clearing wins
for q in my_queries:
    e = acc.explain(q)
    print(f"{e['route']!s:12s} "
          f"ok_dense={e['p_ok_dense_only']:.2f} "
          f"ok_rrf={e['p_ok_pure_rrf']:.2f} "
          f"ok_sparse={e['p_ok_sparse_only']:.2f}  {q}")

dense_only   ok_dense=0.60 ok_rrf=0.52 ok_sparse=0.49  Who likes Curling?
dense_only   ok_dense=0.62 ok_rrf=0.51 ok_sparse=0.44  what are the side effects of DHA
dense_only   ok_dense=1.00 ok_rrf=1.00 ok_sparse=0.00  CVE-2021-44228 log4j remote code execution
pure_rrf     ok_dense=0.44 ok_rrf=0.49 ok_sparse=0.45  http://localhost.com
